# FFT Demo: 32-point Hardware FFT on Pynq

Demonstrates the HLS-synthesised 32-point FFT running on the PL via AXI DMA.

**Data format:** each complex sample is packed into one 32-bit DMA word,
with the real part in bits [15:0] and the imaginary part in bits [31:16],
both stored as `ap_fixed<16,8>`.

To keep the host-side overhead small, this notebook packs a single 32-sample
block with vectorised NumPy code, broadcasts that packed block across the DMA
buffer, and verifies only the first output block against NumPy by default.
If you want, you can enable full all-block verification as well.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from pynq import Overlay, allocate

%matplotlib inline

N_FFT = 32
N_BLOCKS = 1024
FRAC_BITS = 8
SCALE = 1 << FRAC_BITS
VERIFY_ALL_BLOCKS = False

## 1. Load Overlay

In [ ]:
ol = Overlay("fft_design.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma = ol.axi_dma_0

## 2. Test Signal

Two cosines at known bins: bin 4 (amplitude 1.0) and bin 10 (amplitude 0.5).
For a real-valued input the spectrum is symmetric, so we expect peaks at
bins 4, 10, 22, and 28.

In [ ]:
n = np.arange(N_FFT)

x_re = (1.0 * np.cos(2 * np.pi * 4 * n / N_FFT) +
        0.5 * np.cos(2 * np.pi * 10 * n / N_FFT)).astype(np.float32)
x_im = np.zeros(N_FFT, dtype=np.float32)

fig, ax = plt.subplots(figsize=(8, 3))
ax.stem(n, x_re, markerfmt='C0o', linefmt='C0-', basefmt='k-')
ax.set_xlabel('Sample index n')
ax.set_ylabel('Amplitude')
ax.set_title('Input signal x[n]')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Vectorised Fixed-Point Packing

`ap_fixed<16,8>` uses 8 fractional bits, so the scale factor is $2^8 = 256$.
The helpers below operate on whole NumPy arrays instead of one sample at a time.

In [ ]:
def quantize_fixed16(values):
    values = np.asarray(values, dtype=np.float32)
    fixed = np.rint(values * SCALE)
    fixed = np.clip(fixed, -32768, 32767).astype(np.int16)
    return fixed

def pack_complex_samples(re_vals, im_vals):
    re_fixed = quantize_fixed16(re_vals)
    im_fixed = quantize_fixed16(im_vals)
    re_u32 = re_fixed.view(np.uint16).astype(np.uint32)
    im_u32 = im_fixed.view(np.uint16).astype(np.uint32)
    return re_u32 | (im_u32 << 16)

def unpack_complex_samples(words):
    words = np.asarray(words, dtype=np.uint32)
    re = (words & 0xFFFF).astype(np.uint16).view(np.int16).astype(np.float32)
    im = (words >> 16).astype(np.uint16).view(np.int16).astype(np.float32)
    return (re + 1j * im) / SCALE

## 4. Prepare One Packed Block and Broadcast It

Every DMA block uses the same input signal. Instead of re-packing 1024 blocks in
Python loops, we pack one block once and broadcast it across the DMA buffer.

In [ ]:
TOTAL_SAMPLES = N_FFT * N_BLOCKS

in_buf = allocate(shape=(TOTAL_SAMPLES,), dtype=np.uint32)
out_buf = allocate(shape=(TOTAL_SAMPLES,), dtype=np.uint32)
in_blocks = in_buf.reshape(N_BLOCKS, N_FFT)
out_blocks = out_buf.reshape(N_BLOCKS, N_FFT)

t0 = time.perf_counter()
input_block = pack_complex_samples(x_re, x_im)
t_pack = time.perf_counter() - t0

t0 = time.perf_counter()
in_blocks[:] = input_block
t_fill = time.perf_counter() - t0
out_buf.fill(0)

print(f"Pack one {N_FFT}-sample block: {t_pack * 1e6:.1f} us")
print(f"Replicate across {N_BLOCKS} blocks: {t_fill * 1e6:.1f} us")

## 5. Single DMA Transfer and First-Block Check

The FFT IP consumes the full burst as consecutive 32-point blocks. For correctness,
we usually only need to unpack the first output block because every input block was identical.

In [ ]:
t0 = time.perf_counter()
dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

t0 = time.perf_counter()
Y_hw = unpack_complex_samples(out_blocks[0])
t_unpack_first = time.perf_counter() - t0

print(f"DMA transfer ({N_BLOCKS} x {N_FFT}-pt FFT): {t_dma * 1e3:.2f} ms")
print(f"Amortised per block: {t_dma / N_BLOCKS * 1e6:.2f} us")
print(f"Unpack first block only: {t_unpack_first * 1e6:.1f} us")

## 6. Verify Against NumPy

We compare the first block against `numpy.fft.fft`. As a cheap consistency check,
we also confirm that every packed output block matches the first packed output block.

In [ ]:
Y_ref = np.fft.fft(x_re + 1j * x_im)
max_err = np.max(np.abs(Y_hw - Y_ref))
all_blocks_match = bool(np.all(out_blocks == out_blocks[0])) if N_BLOCKS > 1 else True

print(f"Max |HW - numpy| on first block: {max_err:.4f}")
print(f"All packed output blocks identical: {all_blocks_match}")

if VERIFY_ALL_BLOCKS:
    t0 = time.perf_counter()
    Y_all = unpack_complex_samples(out_buf).reshape(N_BLOCKS, N_FFT)
    t_unpack_all = time.perf_counter() - t0
    err_per_block = np.max(np.abs(Y_all - Y_ref[None, :]), axis=1)
    print(f"Unpack all blocks: {t_unpack_all * 1e3:.2f} ms")
    print(f"Max |HW - numpy| across all blocks: {err_per_block.max():.4f}")

print("\nFirst block non-zero bins:")
print(f"  {'bin':>4}  {'|HW|':>8}  {'|ref|':>8}")
for k in range(N_FFT):
    if abs(Y_ref[k]) > 0.1:
        print(f"  {k:4d}  {abs(Y_hw[k]):8.3f}  {abs(Y_ref[k]):8.3f}")

## 7. Plot Magnitude Spectrum

In [ ]:
bins = np.arange(N_FFT)
mag_hw = np.abs(Y_hw)
mag_ref = np.abs(Y_ref)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.stem(bins, mag_ref, markerfmt='C0o', linefmt='C0-', basefmt='k-', label='numpy ref')
ax.stem(bins, mag_hw, markerfmt='C1x', linefmt='C1--', basefmt='k-', label='HW FFT')
ax.set_xlabel('Bin k')
ax.set_ylabel('|X[k]|')
ax.set_title('Magnitude spectrum')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(bins, np.abs(Y_hw - Y_ref), color='C2')
ax.set_xlabel('Bin k')
ax.set_ylabel('|HW - ref|')
ax.set_title(f'Error per bin (max = {max_err:.3f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Compare Against NumPy Runtime on ARM

This uses Python wall-clock timing only. The reported hardware time includes DMA,
software driver overhead, and the FFT itself.

In [ ]:
N_TRIALS = 1000
x_cplx = x_re + 1j * x_im
t0 = time.perf_counter()
for _ in range(N_TRIALS):
    _ = np.fft.fft(x_cplx)
t_np = (time.perf_counter() - t0) / N_TRIALS

print(f"numpy FFT (ARM): {t_np * 1e6:.1f} us per {N_FFT}-pt FFT")
print("\nSummary:")
print(f"  {'Pack one block':<28} {t_pack * 1e6:>10.1f} us")
print(f"  {'Replicate packed block':<28} {t_fill * 1e6:>10.1f} us")
print(f"  {'DMA + FFT':<28} {t_dma * 1e3:>10.2f} ms")
print(f"  {'DMA + FFT per block':<28} {t_dma / N_BLOCKS * 1e6:>10.2f} us")
print(f"  {'Unpack first block':<28} {t_unpack_first * 1e6:>10.1f} us")
print(f"  {'numpy FFT per block':<28} {t_np * 1e6:>10.1f} us")

## 9. Cleanup

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()